# Branch Tracing: Every Mode and Ghost of a Crystal Plate

A sequential trace follows one mode at each surface. A crystal splits every
ray into two transmitted modes and two reflected ones, so the light takes
many paths. `optiland.raytrace.branches.BranchTracer` traces all of them:
each path (a *branch*) is an ordinary fixed-count trace, named by its key,
the list of the choices made at the splitting surfaces. A power threshold and
a limit on the number of reflections end the enumeration, and a power ledger
says where the launch power went.

This example splits the rays of a calcite beam displacer, first into its two
transmitted modes, then with every reflection followed.

In [1]:
import numpy as np

from optiland.materials import IdealMaterial, UniaxialMaterial
from optiland.optic import Optic
from optiland.rays import PolarizationState
from optiland.raytrace.branches import BranchTracer

n_o, n_e = 1.6583434042, 1.4861300612  # calcite at 589.3 nm (Ghosh 1999)
wavelength = 0.5893

calcite = UniaxialMaterial(
    IdealMaterial(n_o), IdealMaterial(n_e), optic_axis=(1.0, 0.0, 1.0)
)
displacer = Optic()
displacer.surfaces.add(index=0, radius=np.inf, thickness=np.inf)
displacer.surfaces.add(
    index=1, thickness=10.0, material=calcite, is_stop=True,
    interaction_type="anisotropic", comment="front",
)  # fmt: skip
displacer.surfaces.add(
    index=2, thickness=5.0, interaction_type="anisotropic", comment="back"
)
displacer.surfaces.add(index=3)
displacer.set_aperture(aperture_type="EPD", value=2.0)
displacer.fields.set_type(field_type="angle")
displacer.fields.add(y=0.0)
displacer.wavelengths.add(value=wavelength, is_primary=True)

azimuth = np.radians(30.0)
displacer.updater.set_polarization(
    PolarizationState(
        is_polarized=True, Ex=np.cos(azimuth), Ey=np.sin(azimuth),
        phase_x=0.0, phase_y=0.0,
    )
)  # fmt: skip

## The transmitted branches

Without ghosts the tracer follows each transmitted mode of each anisotropic
surface. A key entry is `(label, "T", mode)` into a crystal and
`(label, "T")` into an isotropic medium; the label is the surface comment.
Each branch holds its rays after the last surface; `rays.i` is the power of
each ray and `branch.power` the power of the branch as a fraction of the
launch power.

In [2]:
tracer = BranchTracer(displacer)
result = tracer.trace_all(num_rays=19)  # a hexapolar pupil of 19 rays
for key, branch in result.branches.items():
    x = round(float(np.mean(np.asarray(branch.rays.x))), 9) + 0.0
    print(f"{key}: power {branch.power:.9f}, mean x {x:+.6f} mm")

(('front', 'T', 'o'), ('back', 'T')): power 0.220274689, mean x +0.000000 mm
(('front', 'T', 'e'), ('back', 'T')): power 0.678951904, mean x -1.092064 mm


The reflections at the two faces are not followed here; the ledger books
them as `unfollowed`. Calcite is lossless, so nothing is absorbed and the
ledger adds to 1.

In [3]:
for name, value in result.ledger.as_dict().items():
    print(f"{name:>11}: {value:.3e}")
print(f"      total: {result.ledger.total():.15f}")

       kept: 8.992e-01
   returned: 0.000e+00
     pruned: 0.000e+00
 unfollowed: 1.008e-01
 evanescent: 0.000e+00
    clipped: 0.000e+00
   absorbed: 4.982e-17
      total: 1.000000000000000


## Every ghost of the plate

`ghosts="all"` splits every surface into its reflected children as well.
A reflection sends the branch back through the earlier surfaces. With the
threshold at 0 and at most four reflections, the powers of the o and e
branches that reach the image are the terms of the incoherent plate series
T² R^(2m) of each mode, where T = 4n / (1 + n)² and R = 1 − T at normal
incidence (n_o for the o wave, n(45°) for the e wave).

In [4]:
ghosts = BranchTracer(
    displacer, ghosts="all", threshold=0.0, max_reflections=4
).trace_all(num_rays=1, distribution="hexapolar")

t = np.radians(45.0)
n_45 = 1.0 / np.sqrt(np.cos(t) ** 2 / n_o**2 + np.sin(t) ** 2 / n_e**2)
for mode, n, weight in (
    ("o", n_o, np.sin(azimuth) ** 2),
    ("e", n_45, np.cos(azimuth) ** 2),
):
    T = 4 * n / (1 + n) ** 2
    for m in range(3):
        key = (("front", "T", mode),)
        key += (("back", "R", mode), ("front", "R", mode)) * m
        key += (("back", "T"),)
        traced = ghosts[key].power
        closed = weight * T**2 * (1 - T) ** (2 * m)
        print(
            f"{mode}, {m} round trips: {traced:.6e}  closed form {closed:.6e}"
            f"  difference {abs(traced - closed):.1e}"
        )

o, 0 round trips: 2.202747e-01  closed form 2.202747e-01  difference 5.6e-17
o, 1 round trips: 8.285719e-04  closed form 8.285719e-04  difference 7.6e-19
o, 2 round trips: 3.116706e-06  closed form 3.116706e-06  difference 4.7e-21
e, 0 round trips: 6.789519e-01  closed form 6.789519e-01  difference 4.4e-16
e, 1 round trips: 1.599940e-03  closed form 1.599940e-03  difference 1.3e-17
e, 2 round trips: 3.770235e-06  closed form 3.770235e-06  difference 6.5e-20


In [5]:
print(f"{len(ghosts)} branches reach the image or leave backwards")
print(f"returned to object space: {ghosts.ledger.returned:.6f}")
print(f"pruned by the reflection limit: {ghosts.ledger.pruned:.3e}")
print(f"ledger total: {ghosts.ledger.total():.15f}")

22 branches reach the image or leave backwards
returned to object space: 0.098338
pruned by the reflection limit: 3.960e-07
ledger total: 1.000000000000000


## One branch as a sequence

`BranchTracer.sequence(key)` returns a `SequencedOptic` that traces one
branch, so every Optiland analysis that works on a sequence works on it.

In [6]:
e_key = (("front", "T", "e"), ("back", "T"))
e_path = tracer.sequence(e_key)
rays = e_path.trace_generic(0.0, 0.0, 0.0, 0.0, wavelength)
rays.update_intensity(displacer.polarization_state)
print(
    f"e branch, chief ray: x = {float(rays.x[0]):+.9f} mm,"
    f" power = {float(rays.i[0]):.9f}"
)

e branch, chief ray: x = -1.092064213 mm, power = 0.678951904
